# Current single-signal evaluation: Task 1 insertion and Task 2 substitution

Whisper large-v2 / large-v3; official test speakers only. Task 1 excludes C/C/I; Task 2 uses the saved similarity-filtered, word-contrast-complete cohort. All speakers are retained. Cohorts are taken directly from the current six-signal LR test predictions and verified against the aligned feature tables.

Only the current six signals are considered (word contrast only for Task 2). Positive class: **1 = ASR false alarm**. Signal directions are fixed semantically, never selected by test AUC. Confidence, CTC probability and consistency are negated; entropies and prompt-minus-ASR contrast are positive.

Single-signal ROC-AUC, and addition resutls for extra metrics (not included in the paper) positive AP and trapezoidal PR-AUC are recomputed using the same saved 2,000 speaker-bootstrap draws as LR (pooled words, percentile 95% CI). LR results are loaded from the existing six-signal summary; models are not retrained. No demo-data fallback.

Paper numbering: **Task 1 = insertion attribution** (internal column `task2`); **Task 2 = substitution attribution** (internal column `task3`). Only display names are changed; source CSVs and evaluation rules retain their internal names.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'scripts/six_signal_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the project root or notebooks directory')
sys.path.insert(0, str(ROOT / 'scripts'))
from six_signal_config import BASE, WORD
from evaluate_jasmin_cci_only import METRICS, metric_function, reference

SOURCE = ROOT / 'notebooks/jasmin_ctc_contrast_task3'
LR = ROOT / 'notebooks/jasmin_six_signal_lr'
OUT = ROOT / 'notebooks/signal_task_auc_current'
OUT.mkdir(exist_ok=True)
PROTOCOLS = {'task2': 'cci_removed', 'task3': 'filtered_word_common'}
DIRECTIONS = {'vanilla_log_probability': -1, 'lm_entropy': 1,
              'A_ctc': -1, 'audio_entropy': 1,
              'decoding_consistency': -1, WORD: 1}
plan = pd.read_csv(LR / 'bootstrap_speaker_counts.csv')
speakers = list(plan.columns[1:])
counts = plan.iloc[:, 1:].to_numpy(int)
assert len(counts) == 2000 and len(speakers) == 30
assert (counts >= 0).all() and (counts.sum(axis=1) == 30).all()
unit_map = {s: i for i, s in enumerate(speakers)}
pd.set_option('display.max_columns', 30)


## Load and verify the exact test cohorts

Verify word identity, labels, speaker coverage and CCI removal against the saved LR test cohorts. Single signals use the same test words as LR, without additional per-signal row dropping. Sample-count tables are retained internally for validation but are not displayed.

In [2]:
cohorts = {}
cohort_rows = []
total_rows = []
for variant in ['v2', 'v3']:
    features = pd.read_csv(SOURCE / f'{variant}_test_aligned.csv', low_memory=False)
    assert not features.duplicated(['speaker', 'asr_index']).any()
    total_rows.append(dict(variant=variant, split='test',
                           n_all_input_word_rows=len(features),
                           n_speakers=features.speaker.nunique(),
                           n_task2_before_filter=int(features.task2.notna().sum()),
                           n_task3_before_filter=int(features.task3_original.notna().sum())))
    original_all_c = features.pr_mt_labels.fillna('').map(
        lambda x: bool(str(x).strip()) and all(t.strip().upper() == 'C' for t in str(x).split('|')))
    original_cci = original_all_c & features.mt_v_label.eq('C') & features.pr_v_label.eq('I')
    for task, protocol in PROTOCOLS.items():
        pred = pd.read_csv(LR / f'{variant}_{task}_{protocol}_test_scores.csv')
        f = pred[['speaker', 'asr_index', 'asr_word', 'y_true']].merge(
            features, on=['speaker', 'asr_index'], how='left',
            validate='one_to_one', suffixes=('_prediction', ''))
        assert len(f) == len(pred)
        assert f.asr_word.eq(f.asr_word_prediction).all()
        assert f.y_true.eq(f[task]).all()
        assert set(f.speaker) == set(speakers)
        assert set(f.y_true.unique()) == {0, 1}
        conf = f.lm_confidence.to_numpy(float)
        assert np.isfinite(conf).all() and (conf > 0).all()
        f['vanilla_log_probability'] = np.log(conf)
        selected = BASE + ([WORD] if task == 'task3' else [])
        assert np.isfinite(f[selected].to_numpy(float)).all()
        all_c = f.pr_mt_labels.fillna('').map(
            lambda x: bool(str(x).strip()) and all(t.strip().upper() == 'C' for t in str(x).split('|')))
        cci = all_c & f.mt_v_label.eq('C') & f.pr_v_label.eq('I')
        if task == 'task2':
            assert not cci.any()
        cohorts[(variant, task)] = f
        original = features.task2.notna() if task == 'task2' else features.task3_original.notna()
        after_filter = (original & ~original_cci) if task == 'task2' else features.task3.notna()
        assert not (after_filter & ~original).any()
        n_before = int(original.sum())
        n_after = int(after_filter.sum())
        expected = features.loc[after_filter, ['speaker', 'asr_index']].merge(
            f[['speaker', 'asr_index']], on=['speaker', 'asr_index'], validate='one_to_one')
        assert len(expected) == len(f) and n_before >= n_after >= len(f)
        cohort_rows.append(dict(variant=variant, task=task, protocol=protocol,
                                n_task_before_filter=n_before,
                                n_removed_by_cci_or_similarity=n_before-n_after,
                                n_after_task_filter=n_after,
                                n_removed_by_feature_coverage=n_after-len(f),
                                n_words=len(f), n_removed_total=n_before-len(f),
                                n_speakers=f.speaker.nunique(),
                                n_positive=int(f.y_true.sum()),
                                n_negative=int((f.y_true == 0).sum())))

cohort_summary = pd.DataFrame(cohort_rows)
input_totals = pd.DataFrame(total_rows)
input_totals.to_csv(OUT / 'input_sample_totals.csv', index=False)


## Single-signal metrics and shared speaker-bootstrap intervals

In [3]:
rows = []
for (variant, task), f in cohorts.items():
    y = f.y_true.to_numpy(int)
    units = f.speaker.map(unit_map).to_numpy(int)
    for signal in BASE + ([WORD] if task == 'task3' else []):
        score = f[signal].to_numpy(float) * DIRECTIONS[signal]
        evaluate = metric_function(y, score, units)
        point = evaluate(np.ones(len(speakers), dtype=int))
        np.testing.assert_allclose(point, reference(y, score), atol=1e-12)
        draws = np.array([evaluate(draw) for draw in counts])
        for k in [0, 17]:
            np.testing.assert_allclose(draws[k], reference(y, score, counts[k][units]), atol=1e-12)
        row = dict(variant=variant, task=task, protocol=PROTOCOLS[task],
                   signal=signal, multiplier=DIRECTIONS[signal], n_test=len(f))
        for j, metric in enumerate(METRICS):
            valid = draws[:, j][np.isfinite(draws[:, j])]
            low, high = np.quantile(valid, [.025, .975])
            row.update({metric: point[j], metric+'_ci_lower': low,
                        metric+'_ci_upper': high, metric+'_n_valid': len(valid)})
        rows.append(row)
single_summary = pd.DataFrame(rows)
assert len(single_summary) == 22
single_summary.to_csv(OUT / 'single_signal_summary.csv', index=False)
cohort_summary.to_csv(OUT / 'cohort_counts.csv', index=False)

def formatted_metrics(table, name):
    view = table[['variant', 'task', name]].copy()
    # Paper numbering only: leave data columns and evaluation logic unchanged.
    view['task'] = view['task'].map({'task2': 'Task 1', 'task3': 'Task 2'})
    for metric in METRICS:
        view[metric + ' [95% CI]'] = table.apply(
            lambda r: f"{r[metric]:.4f} [{r[metric+'_ci_lower']:.4f}, {r[metric+'_ci_upper']:.4f}]", axis=1)
    return view

display(formatted_metrics(single_summary, 'signal'))


,variant,task,signal,roc_auc [95% CI],ap_positive [95% CI],pr_auc_positive [95% CI]
0,v2,Task 1,vanilla_log_probability,"0.7203 [0.6522, 0.7905]","0.1359 [0.0944, 0.2089]","0.1310 [0.0876, 0.1987]"
1,v2,Task 1,lm_entropy,"0.7113 [0.6358, 0.7903]","0.1239 [0.0879, 0.1804]","0.1208 [0.0831, 0.1731]"
2,v2,Task 1,A_ctc,"0.6976 [0.6376, 0.7668]","0.1621 [0.1125, 0.2432]","0.1776 [0.1197, 0.2553]"
3,v2,Task 1,audio_entropy,"0.6809 [0.5819, 0.7542]","0.1529 [0.0755, 0.2789]","0.1488 [0.0702, 0.2717]"
4,v2,Task 1,decoding_consistency,"0.6475 [0.5413, 0.7553]","0.1257 [0.0823, 0.1900]","0.1563 [0.1014, 0.2291]"
5,v2,Task 2,vanilla_log_probability,"0.6511 [0.6109, 0.6906]","0.7684 [0.7086, 0.8297]","0.7678 [0.7071, 0.8290]"
6,v2,Task 2,lm_entropy,"0.6387 [0.5979, 0.6776]","0.7685 [0.7091, 0.8291]","0.7680 [0.7078, 0.8281]"
7,v2,Task 2,A_ctc,"0.5515 [0.4843, 0.6203]","0.7344 [0.6211, 0.8208]","0.7346 [0.6190, 0.8211]"
8,v2,Task 2,audio_entropy,"0.5375 [0.4809, 0.5950]","0.7227 [0.6312, 0.8007]","0.7223 [0.6298, 0.8001]"
9,v2,Task 2,decoding_consistency,"0.6461 [0.6050, 0.6879]","0.7598 [0.7054, 0.8130]","0.7739 [0.7198, 0.8247]"


## Current LR results (saved, primary C = 0.1)

Only Task 1 CCI-removed and Task 2 filtered-word-common results are shown. These are loaded from the existing six-signal evaluation, not refitted in this notebook.

In [4]:
lr_summary = pd.read_csv(LR / 'summary.csv')
keep = lr_summary.C.eq(.1) & (
    (lr_summary.task.eq('task2') & lr_summary.protocol.eq(PROTOCOLS['task2'])) |
    (lr_summary.task.eq('task3') & lr_summary.protocol.eq(PROTOCOLS['task3'])))
lr_current = lr_summary.loc[keep].copy()
assert len(lr_current) == 20
for (variant, task), f in cohorts.items():
    assert lr_current.loc[lr_current.variant.eq(variant) & lr_current.task.eq(task), 'n_test'].eq(len(f)).all()
display(formatted_metrics(lr_current, 'method'))
lr_current.to_csv(OUT / 'lr_primary_summary.csv', index=False)
print('Updated outputs:', OUT)


,variant,task,method,roc_auc [95% CI],ap_positive [95% CI],pr_auc_positive [95% CI]
8,v2,Task 1,lr_vanilla_C0.1,"0.7205 [0.6457, 0.7967]","0.1310 [0.0918, 0.1952]","0.1279 [0.0868, 0.1879]"
10,v2,Task 1,lr_vanilla_plus_ctc_mean_C0.1,"0.7560 [0.6880, 0.8309]","0.2082 [0.1528, 0.3013]","0.2042 [0.1469, 0.2918]"
12,v2,Task 1,lr_vanilla_plus_decoding_consistency_C0.1,"0.7264 [0.6324, 0.8127]","0.1463 [0.1046, 0.2137]","0.1427 [0.0991, 0.2037]"
14,v2,Task 1,lr_all_without_contrast_C0.1,"0.7593 [0.6695, 0.8500]","0.1996 [0.1339, 0.3193]","0.1932 [0.1245, 0.3066]"
16,v2,Task 2,lr_vanilla_C0.1,"0.6491 [0.6091, 0.6882]","0.7699 [0.7099, 0.8309]","0.7694 [0.7080, 0.8303]"
18,v2,Task 2,lr_vanilla_plus_ctc_mean_C0.1,"0.6403 [0.5919, 0.6927]","0.7731 [0.6983, 0.8434]","0.7727 [0.6967, 0.8430]"
20,v2,Task 2,lr_vanilla_plus_decoding_consistency_C0.1,"0.6686 [0.6273, 0.7110]","0.7790 [0.7238, 0.8332]","0.7786 [0.7225, 0.8323]"
22,v2,Task 2,lr_all_without_contrast_C0.1,"0.6692 [0.6250, 0.7156]","0.7867 [0.7232, 0.8442]","0.7864 [0.7218, 0.8437]"
24,v2,Task 2,lr_vanilla_plus_word_contrast_C0.1,"0.7097 [0.6800, 0.7435]","0.8300 [0.7889, 0.8677]","0.8299 [0.7884, 0.8675]"
26,v2,Task 2,lr_all_with_word_contrast_C0.1,"0.7237 [0.6913, 0.7591]","0.8348 [0.7917, 0.8732]","0.8346 [0.7912, 0.8729]"


Updated outputs: /vol/tensusers/lgao/uncertainty_whisper/0_uncertainty_attribution_whisper/notebooks/signal_task_auc_current
